# `ward` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `high-cardinality-category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'ward'
feature_metadata = {'order': 16, 'name': 'ward', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'retain with LGA context and fold-fitted rare grouping', 'finding': 'Raw unseen exposure is small, but many ward names are reused and low-frequency.', 'decision': 'Prefer an LGA/ward composite or encoder with explicit rare and unseen handling.', 'risk': 'Raw ward names can be ambiguous outside their administrative context.', 'related': [{'feature': 'lga', 'reason': 'LGA context disambiguates reused ward names.'}, {'feature': 'subvillage', 'reason': 'Subvillages are the finer named location.'}, {'feature': 'region', 'reason': 'Region provides a broad back-off.'}, {'feature': 'longitude', 'reason': 'Coordinates provide an independent location representation.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for ward.


## Supported target evidence


In [2]:
sentinel_tokens = []
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
ward,,,,,
igosi,307,True,94.14,0.00,5.86
imalinyi,252,True,95.24,1.19,3.57
siha kati,232,True,98.28,0.86,0.86
mdandu,231,True,87.45,5.63,6.93
nduruma,217,True,63.13,7.37,29.49
kitunda,203,True,79.31,0.00,20.69
mishamo,203,True,21.18,7.39,71.43
msindo,201,True,69.15,6.97,23.88
chalinze,196,True,78.06,0.00,21.94


status_group,rows,non functional (%)
ward,,
mishamo,203,71.43
kikatiti,134,54.48
tinde,106,45.28
makwale,104,45.19
mkongo,107,44.86
mvomero,129,44.19
ifakara,134,44.03
diongoya,103,41.75
simbo,118,41.53


## Observation

Raw unseen exposure is small, but many ward names are reused and low-frequency.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Prefer an LGA/ward composite or encoder with explicit rare and unseen handling.

**Risk to carry forward:** Raw ward names can be ambiguous outside their administrative context.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
ward,candidate,retain with LGA context and fold-fitted rare g...,"Raw unseen exposure is small, but many ward na...",Prefer an LGA/ward composite or encoder with e...,Raw ward names can be ambiguous outside their ...
